# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR^2 dataset using the `mlcroissant` library, referencing all entities by their `@id` as specified in the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** All references use their Croissant `@id` values.

In [ ]:
# List all available record sets, their @ids and schema details
record_sets = dataset.record_sets
print(f"Record sets in the dataset ({len(record_sets)}):\n")
for rs in record_sets:
    print(f"- Name: {rs.name}")
    print(f"  @id: {rs.id}")
    field_ids = [f.id for f in rs.fields]
    print(f"  Fields: {field_ids}\n")

# For illustration, extract the first record set to inspect its records and field info
if record_sets:
    rs0 = record_sets[0]
    print(f"First record set: {rs0.name}\n@id: {rs0.id}\nFields:")
    for field in rs0.fields:
        print(f"  - {field.name} (@id: {field.id}, dataType: {field.data_type})")
    print("\nFirst few records:")
    for ix, record in enumerate(dataset.records(record_set=rs0.id)):
        print(record)
        if ix >= 2: break

## 3. Data Extraction
Load data from all main record sets into pandas DataFrames for analysis. Use record set and field `@id`s for references.

In [ ]:
# Extract data from each available record set using @id
dataframes = {}
for rs in dataset.record_sets:
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"DataFrame for record set '{rs.name}' (@id: {rs.id}): {df.shape} rows, columns: {df.columns.tolist()}")

# Display head of the first main record set (assumed primary data)
if dataset.record_sets:
    main_rs_id = dataset.record_sets[0].id
    print(f"\nFirst few rows from main record set (@id={main_rs_id}):")
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing: filtering, normalizing numeric fields, and grouping by key categorical variables. 

All columns/fields referenced use their Croissant `@id`.

In [ ]:
# EDA on primary record set
if dataset.record_sets:
    rs = dataset.record_sets[0]
    df = dataframes[rs.id].copy()
    
    # Identify a numeric field by @id (example: 'cr:coverage')
    numeric_fields = [f for f in rs.fields if f.data_type in ('Float', 'Integer', 'Number')]
    if numeric_fields:
        numeric_field_id = numeric_fields[0].id
        threshold = 10
        print(f"Using numeric field: {numeric_field_id} (dataType: {numeric_fields[0].data_type})")
        df_num = pd.to_numeric(df[numeric_field_id], errors='coerce')
        
        # Filtering records
        filtered_df = df[df_num > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}: {filtered_df.shape[0]} records")
        display(filtered_df.head())

        # Normalizing
        filtered_df[f"{numeric_field_id}_normalized"] = (df_num - df_num.mean()) / df_num.std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a categorical field (e.g., 'cr:sample_id' or first String field)
        group_field = None
        for f in rs.fields:
            if f.data_type == 'Text' and f.id != numeric_field_id and f.id in filtered_df:
                group_field = f.id
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped average {numeric_field_id} by {group_field} (first 5 rows):")
            display(grouped_df.head())
        else:
            print("No suitable Text field for grouping found.")
    else:
        print("No numeric field found in the primary record set.")

## 5. Visualization
Visualize data distributions using field `@id` as axis labels.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataset.record_sets and numeric_fields:
    field_id = numeric_field_id
    plt.figure(figsize=(8, 4))
    sns.histplot(pd.to_numeric(df[field_id], errors='coerce').dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {field_id}")
    plt.xlabel(field_id)
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()

    # If a group_field exists, boxplot by group
    if group_field:
        plt.figure(figsize=(10, 4))
        sns.boxplot(data=filtered_df, x=group_field, y=field_id)
        plt.title(f"{field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 mass spectrometry dataset using the `mlcroissant` library. We programmatically referenced record sets, fields, and columns by their Croissant `@id` for reproducibility. Basic exploratory analysis and visualizations highlighted the structure and potential of the dataset for downstream applications such as protein expression analysis and biomarker discovery.